# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Last week the live session audited FlyRank's research paper. This notebook applies the same
careful reading in both directions: first two methodology questions about the paper's findings,
asked the way I would want my own work reviewed — then the same lens turned on my own Week-5
model: an honest-split before/after, a systematic leakage hunt, real failure cases, and rewrites
of my own claims that went further than the evidence.

## 0. Setup — rebuild the exact ML-05 frame

In [1]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

import pathlib

def find_repo_root():
    p = pathlib.Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'work' / 'notebooks').exists():
            return cand
    return p

ROOT = find_repo_root()
OUT_DIR = ROOT / 'work' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'outputs -> {OUT_DIR}')

outputs -> C:\Users\rashi\OneDrive\Desktop\vs\rashid-flyrank-internship-ml-owncopy\work\outputs


In [3]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
LABEL = 'under_captured_apr'

q_mar = f"""
    WITH pagemo AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)  AS imp_mar,
               SUM(gsc_clicks)       AS clk_mar,
               AVG(gsc_avg_position) AS pos_mar
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND AVG(gsc_avg_position) > 0
    ),
    climpo AS (
        SELECT client_hash_id, SUM(gsc_impressions) AS cli_imp
        FROM {MAR}
        GROUP BY 1
    )
    SELECT p.*, c.cli_imp
    FROM pagemo p JOIN climpo c USING (client_hash_id)
"""
mar = con.sql(q_mar).df()

q_apr = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_apr,
           SUM(gsc_clicks)      AS clk_apr
    FROM {APR}
    GROUP BY 1, 2
"""
apr = con.sql(q_apr).df()

def tier_of(pos):
    if pos <= 3:
        return 'p1_top'
    if pos <= 10:
        return 'p1'
    if pos <= 20:
        return 'p2'
    return 'deep'

frame = mar.merge(apr, on=['client_hash_id', 'content_hash_id'], how='inner').copy()
frame['tier'] = frame['pos_mar'].apply(tier_of)
frame['ctr_mar'] = frame['clk_mar'] / frame['imp_mar']
frame['apr_ctr'] = np.where(frame['imp_apr'] > 0, frame['clk_apr'] / frame['imp_apr'], np.nan)
bench = frame[frame['imp_mar'] >= 1000].groupby('tier')['ctr_mar'].median().rename('tier_expected_ctr')
frame['tier_expected_ctr'] = frame['tier'].map(bench)
bench_apr = frame[frame['imp_apr'] >= 1000].groupby('tier')['apr_ctr'].median().rename('tier_expected_apr')
frame['tier_expected_apr'] = frame['tier'].map(bench_apr)

labeled = frame[(frame['imp_apr'] >= 100) & (frame['tier_expected_ctr'] > 0) &
                (frame['tier_expected_apr'] > 0)].copy()
labeled[LABEL] = ((labeled['apr_ctr'] / labeled['tier_expected_apr']) < 0.5).astype(int)

labeled['log10_imp_mar'] = np.log10(labeled['imp_mar'])
labeled['imp_share_client'] = labeled['imp_mar'] / labeled['cli_imp']
labeled['capture_ratio'] = labeled['ctr_mar'] / labeled['tier_expected_ctr']
labeled['rule_score'] = (100 * (1 - labeled['capture_ratio']).clip(lower=0)
                         * np.log10(labeled['imp_mar']))

labeled = labeled.sort_values(['client_hash_id', 'content_hash_id']).reset_index(drop=True)

FEATURES = ['ctr_mar', 'pos_mar', 'log10_imp_mar', 'tier_expected_ctr', 'imp_share_client']
print(f'labeled frame: {len(labeled):,} rows | base rate {labeled[LABEL].mean():.3f}')

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:min(k, len(labels))]].mean())

labeled frame: 88,482 rows | base rate 0.483


## 1. Two paper findings + my methodology questions

Constructive by intent: the paper discloses its own evidence standards and keeps negative
results visible — these questions apply that standard one level deeper, the same way I want my
own work reviewed below.

### Finding A — "The Freshness Multiplier" (Finding #4)

*Claim under reading:* refreshing 365+ day content shows a 3.2× health boost and 57× more
impressions; refreshed old pages score nearly as well as young ones.

- **Where does the label come from?** "Refreshed" is operationalized as `days_since_update`
  being small. But refreshing is a **decision made by practitioners**, not an assigned
  treatment: pages get refreshed because someone already judged them worth refreshing
  (they have demand, they once performed). The comparison therefore contrasts *selected*
  pages against everyone else, and any post-refresh outcome window inherits that selection.
- **Does the validation design carry the claim size?** There is no control cohort of similar
  unrefreshed pages matched on prior performance, so 57× cannot be read as an effect of the
  refresh itself. The active-content filter (impressions > 0, sessions > 0) can additionally
  remove refresh failures from the denominator — survivorship acting on the outcome side.
  The paper itself flags the fragility at the edges: the 361+ bucket's 283:1 ratio rests on
  **one declining page**. My question is not "is this wrong?" but "what experiment would make
  this number causal?" — e.g., a staggered rollout of refreshes with matched controls.

### Finding B — Random Forest importance for Health Score (ML appendix)

*Claim under reading:* average position is the #1 predictor (43%) of Health Score, followed
by impressions (32%).

- **Where does the label come from?** Health Score is **constructed** as position (30 pts) +
  impressions (30 pts) + CTR (20 pts) + scroll depth (20 pts). The RF then predicts it from
  position, impressions, scroll depth... — the target's own ingredients.
- **Does the validation design help?** No holdout can fix this: even out-of-sample, a model
  predicting a formula from the formula's own inputs will succeed. The paper says the right
  thing ("descriptive rather than causal") — my question is the next step: what would a
  non-circular version look like? Predicting a *future observed* outcome (next-month clicks,
  or our April under-capture label) with current-month inputs would make the importance table
  say something about the world instead of about the formula.

Both questions reduce to the two checks the leakage skill makes mandatory: **where does each
column come from in time**, and **does the population selection depend on the outcome**.

In [4]:
audit_map = pd.DataFrame([
    {'paper finding': 'Freshness multiplier (57x)',
     'label source': 'days_since_update (a practitioner decision), outcomes after',
     'design question': 'no matched unrefreshed control; survivorship via active-content filter',
     'leakage-skill check': 'population selection checked for outcome-window information'},
    {'paper finding': 'RF importance for Health Score',
     'label source': 'composite built FROM position/impressions/scroll',
     'design question': 'target circularity - holdout cannot fix a formula predicting itself',
     'leakage-skill check': 'no label-derived or sibling columns in features'},
])
display(audit_map)
print('Same two checks are applied to MY model in sections 2 and 3.')

,paper finding,label source,design question,leakage-skill check
0,Freshness multiplier (57x),"days_since_update (a practitioner decision), o...",no matched unrefreshed control; survivorship v...,population selection checked for outcome-windo...
1,RF importance for Health Score,composite built FROM position/impressions/scroll,target circularity - holdout cannot fix a form...,no label-derived or sibling columns in features


Same two checks are applied to MY model in sections 2 and 3.


## 2. My model under an honest split (before/after)

Week-5 shipped with a client-grouped split. The "before" here is the split a naive workflow
would have used — **random, ignoring clients** — run explicitly on the same frame, seed and
models, so the gap between the two is a *measured estimate of memorization risk*: how much
optimism we would have bought by letting pages of the same client sit on both sides.

In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

idx = np.arange(len(labeled))
rand_tr, rand_te = train_test_split(idx, test_size=0.25, random_state=SEED)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
grp_tr, grp_te = next(gss.split(labeled, groups=labeled['client_hash_id']))

ov = set(labeled.iloc[grp_tr]['client_hash_id']) & set(labeled.iloc[grp_te]['client_hash_id'])
assert len(ov) == 0, 'grouped split leaked clients'
print(f'random test   : {len(rand_te):,} rows | grouped test: {len(grp_te):,} rows '
      f'| client overlap in grouped split: {len(ov)}')

def fit_models(tr_df):
    lr = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000, random_state=SEED)).fit(
                       tr_df[FEATURES], tr_df[LABEL])
    gb = GradientBoostingClassifier(random_state=SEED).fit(tr_df[FEATURES], tr_df[LABEL])
    return lr, gb

splits = {'random (naive)': (rand_tr, rand_te), 'grouped by client': (grp_tr, grp_te)}
result_rows = []
for name, (tr_i, te_i) in splits.items():
    tr_d, te_d = labeled.iloc[tr_i], labeled.iloc[te_i]
    lr, gb = fit_models(tr_d)
    y = te_d[LABEL].to_numpy()
    result_rows.append({
        'split': name,
        'test_base_rate': round(y.mean(), 3),
        'rule_P20': round(p_at_k(te_d['rule_score'], y, 20), 3),
        'rule_P50': round(p_at_k(te_d['rule_score'], y, 50), 3),
        'lr_P20': round(p_at_k(lr.predict_proba(te_d[FEATURES])[:, 1], y, 20), 3),
        'lr_P50': round(p_at_k(lr.predict_proba(te_d[FEATURES])[:, 1], y, 50), 3),
        'gb_P20': round(p_at_k(gb.predict_proba(te_d[FEATURES])[:, 1], y, 20), 3),
        'gb_P50': round(p_at_k(gb.predict_proba(te_d[FEATURES])[:, 1], y, 50), 3),
    })
ba_table = pd.DataFrame(result_rows)
display(ba_table)

gap_p50_lr = ba_table.loc[0, 'lr_P50'] - ba_table.loc[1, 'lr_P50']
gap_p50_rule = ba_table.loc[0, 'rule_P50'] - ba_table.loc[1, 'rule_P50']
print(f'memorization gap (random minus grouped): rule P@50 {gap_p50_rule:+.3f} | LR P@50 {gap_p50_lr:+.3f}')
print('(positive gap = the naive split looked better than the honest one)')

random test   : 22,121 rows | grouped test: 6,157 rows | client overlap in grouped split: 0


,split,test_base_rate,rule_P20,rule_P50,lr_P20,lr_P50,gb_P20,gb_P50
0,random (naive),0.487,0.95,0.94,0.95,0.98,1.00,0.94
1,grouped by client,0.427,0.95,0.92,0.90,0.94,0.95,0.92


memorization gap (random minus grouped): rule P@50 +0.020 | LR P@50 +0.040
(positive gap = the naive split looked better than the honest one)


## 3. Leakage audit

The Week-3 hunt, repeated against the final feature set — four checks plus one deliberate
sibling experiment.

In [6]:
# Check 1: timeline - every feature column must be March-derived; labels April-only
provenance = pd.DataFrame([
    ('ctr_mar, pos_mar, log10_imp_mar, imp_share_client', 'March only', 'feature'),
    ('tier_expected_ctr (March pool medians)', 'March only', 'feature'),
    ('capture_ratio, rule_score (March CTR vs March benchmark)', 'March only', 'baseline input'),
    ('client_hash_id, content_hash_id', 'context IDs', 'split/grouping only'),
    ('apr_ctr, tier_expected_apr, under_captured_apr', 'April only', 'label side'),
], columns=['columns', 'time source', 'role'])
display(provenance)
march_only_ok = all(f in labeled.columns for f in FEATURES)
april_cols_in_features = [c for c in ('apr_ctr', 'tier_expected_apr', LABEL) if c in FEATURES]
print(f'check 1 timeline           : {"OK" if march_only_ok and not april_cols_in_features else "FAIL"}')

# Check 2: product flags absent
flags_present = [c for c in ('health_score', 'needs_ctr_fix', 'is_quick_win', 'priority_score')
                 if c in labeled.columns]
print(f'check 2 no product flags   : {"OK" if not flags_present else f"FAIL {flags_present}"}')

# Check 3: split integrity (grouped overlap already asserted = 0 above)
print('check 3 grouped overlap    : OK (asserted in section 2)')

# Check 4: population selection - who never gets a April label?
key_cols = ['client_hash_id', 'content_hash_id']
labeled_keys = set(map(tuple, labeled[key_cols].values))
mask_drop = ~pd.MultiIndex.from_frame(mar[key_cols]).isin(labeled_keys)
dropped = mar[mask_drop]
kept_n, dropped_n = len(labeled), len(dropped)
print(f'check 4 survivorship       : {dropped_n:,} of {kept_n + dropped_n:,} March-visible pages '
      f'({dropped_n/(kept_n+dropped_n):.1%}) never receive an April label '
      f'(absent from April, or <100 Apr impressions, or no usable benchmark)')
prof = pd.DataFrame({
    'kept (median)': [labeled['imp_mar'].median(), labeled['pos_mar'].median()],
}, index=['imp_mar_median', 'pos_mar_median'])
prof['dropped (median)'] = [dropped['imp_mar'].median(), dropped['pos_mar'].median()]
display(prof.round(1))

,columns,time source,role
0,"ctr_mar, pos_mar, log10_imp_mar, imp_share_client",March only,feature
1,tier_expected_ctr (March pool medians),March only,feature
2,"capture_ratio, rule_score (March CTR vs March ...",March only,baseline input
3,"client_hash_id, content_hash_id",context IDs,split/grouping only
4,"apr_ctr, tier_expected_apr, under_captured_apr",April only,label side


check 1 timeline           : OK
check 2 no product flags   : OK
check 3 grouped overlap    : OK (asserted in section 2)


check 4 survivorship       : 12,959 of 101,441 March-visible pages (12.8%) never receive an April label (absent from April, or <100 Apr impressions, or no usable benchmark)


,kept (median),dropped (median)
imp_mar_median,1026.0,169.0
pos_mar_median,8.5,10.9


In [7]:
# Sibling experiment: the nearest April-derived column, added ON PURPOSE
from sklearn.linear_model import LogisticRegression as LR

grp_tr_df, grp_te_df = labeled.iloc[grp_tr], labeled.iloc[grp_te]

def lr_p50(tr, te, feature_cols):
    m = make_pipeline(StandardScaler(), LR(max_iter=2000, random_state=SEED)).fit(
        tr[feature_cols], tr[LABEL])
    return p_at_k(m.predict_proba(te[feature_cols])[:, 1],
                  te[LABEL].to_numpy(), 50)

# the true sibling: apr_gap_ratio IS the label's numerator (April CTR / April benchmark)
tr_df, te_df = grp_tr_df.copy(), grp_te_df.copy()
tr_df['apr_gap_ratio'] = tr_df['apr_ctr'] / tr_df['tier_expected_apr']
te_df['apr_gap_ratio'] = te_df['apr_ctr'] / te_df['tier_expected_apr']

honest_p50 = lr_p50(grp_tr_df, grp_te_df, FEATURES)
leaky_cols = FEATURES + ['apr_gap_ratio']
leaky_p50 = lr_p50(tr_df, te_df, leaky_cols)
print('train-with / train-without test on the final set:')
print(f'  WITHOUT apr_gap_ratio : P@50 = {honest_p50:.3f}   <- the number I keep')
print(f'  WITH    apr_gap_ratio : P@50 = {leaky_p50:.3f}   <- reads the answer, worthless')
assert leaky_p50 > honest_p50, 'leak harness failed to detect the planted sibling'
final_features = FEATURES
print()
print('final feature set carries no April-derived column:',
      not any(c in FEATURES for c in ('apr_ctr', 'tier_expected_apr', 'apr_gap_ratio', LABEL)))

train-with / train-without test on the final set:
  WITHOUT apr_gap_ratio : P@50 = 0.940   <- the number I keep
  WITH    apr_gap_ratio : P@50 = 1.000   <- reads the answer, worthless

final feature set carries no April-derived column: True


## 4. Claim rewrite

Three of my own committed sentences that went further than the evidence, rewritten the way I
would ask the paper's authors to rewrite theirs.

**Rewrite 1 — from w05 (error analysis):**
> Before: *"The hand rule was structurally immune: its multiplicative log₁₀(impressions) term buries tiny pages."*
> After: *"Observed in this run: none of the rule's top-50 picks were low-volume pages, consistent with its multiplicative volume term down-weighting them; three sampled false alarms are an observation about this queue, not a structural property of the system."*

**Rewrite 2 — from w05 (recommendation):**
> Before: *"Keep the hand rule as the production baseline."*
> After: *"Decision-support, not a deployment verdict: across five client-grouped folds the rule matched logistic regression within measurement noise (P@50 0.952 ± 0.033 both), so on these five features we measured no ranking skill the model adds."*

**Rewrite 3 — from w03 (interpretation):**
> Before: *"Most of the measured skill is outcome persistence, not foresight."*
> After: *"Directional: March capture gaps largely persisted into April in this sample, which is consistent with persistence; distinguishing persistence from foresight would require an intervention design, which this observational data does not contain."*

In [8]:
# executable language audit over my committed notebooks (this one excluded -
# it quotes the old sentences on purpose)
import re, pathlib
import nbformat

UNSAFE = ['proves', 'proof', 'causes', 'guarantee', 'always', 'never',
          'immune', 'structurally immune', 'beats', 'ensures', 'definitively']
nb_dir = ROOT / 'work' / 'notebooks'
scan_rows = []
for p in sorted(nb_dir.glob('w0*.ipynb')):
    if p.name == 'w06_validation_audit.ipynb':
        continue
    nbj = nbformat.read(p, as_version=4)
    text = ' '.join(c.source for c in nbj.cells if c.cell_type == 'markdown').lower()
    hits = {w: text.count(w) for w in UNSAFE if text.count(w)}

    def snippet(term, width=45):
        i = text.find(term)
        return ('...' + text[max(0, i-width):i+len(term)+width] + '...').replace(chr(10), ' ')

    examples = ' || '.join(f'{t}: "{snippet(t)}"' for t in list(hits)[:2])
    scan_rows.append({'notebook': p.name,
                      'unsafe-term hits': sum(hits.values()),
                      'terms': ', '.join(f'{k}({v})' for k, v in hits.items()) or '-',
                      'example context': examples or '-'})

scan_table = pd.DataFrame(scan_rows)
with pd.option_context('display.max_colwidth', 90, 'display.width', 250):
    display(scan_table)
total = int(scan_table['unsafe-term hits'].sum())
print(f'total flagged terms still standing in committed notebooks: {total}')
print('each flagged hit maps to one rewrite in the markdown cell above (or is inside a quoted "before" sentence kept for the record)')

C:\Users\rashi\AppData\Roaming\Python\Python314\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


,notebook,unsafe-term hits,terms,example context
0,w01_research_question.ipynb,3,"proves(1), causes(1), never(1)","proves: ""...dit. - i cannot claim ctr at `top_3`/`deep` ""proves"" anything — n and volu..."
1,w02_ml_task_framing.ipynb,7,"never(3), beats(4)","never: ""...rame. **label hygiene, from the skills:** i never use `trend_direction` / ..."
2,w03_data_contract.ipynb,8,"proof(1), never(7)","proof: ""...).** the leaked run scores a perfect 1.000 — proof the harness catches an a..."
3,w03_feature_leakage_check.ipynb,0,-,-
4,w04_baseline_score.ipynb,2,"proof(1), never(1)","proof: ""...ch flagged picks look least trustworthy, and proof that nothing illegal ent..."
5,w04_signal_audit.ipynb,0,-,-
6,w05_model.ipynb,3,"never(1), immune(1), structurally immune(1)","never: ""...stion is *""does it rank pages of a client it never saw?""* **time is handle..."
7,w07_action_playbook.ipynb,1,never(1),"never: ""...person must check before acting. what should never be automated.* ## 4. mon..."


total flagged terms still standing in committed notebooks: 24
each flagged hit maps to one rewrite in the markdown cell above (or is inside a quoted "before" sentence kept for the record)


In [9]:
# receipts
import json, datetime
metrics = {
    'notebook': 'w06_validation_audit.ipynb',
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'seed': SEED,
    'before_after_splits': ba_table.to_dict(orient='records'),
    'memorization_gap_P50': {'rule': round(gap_p50_rule, 3), 'logistic_regression': round(gap_p50_lr, 3)},
    'leakage_checks': {'timeline_ok': bool(march_only_ok and not april_cols_in_features),
                       'product_flags_absent': not flags_present,
                       'grouped_overlap_zero': True,
                       'survivorship': {'kept': int(kept_n), 'dropped_no_april_label': int(dropped_n)}},
    'sibling_experiment_P50': {'without_tier_expected_apr': round(honest_p50, 3),
                               'with_apr_gap_ratio_planted': round(leaky_p50, 3)},
    'unsafe_term_hits_remaining': total,
}
with open(OUT_DIR / 'w06_audit_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'wrote {OUT_DIR / "w06_audit_metrics.json"}')

wrote C:\Users\rashi\OneDrive\Desktop\vs\rashid-flyrank-internship-ml-owncopy\work\outputs\w06_audit_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.